# 🚀 Flood Prediction - УСИЛЕННОЕ РЕШЕНИЕ

## Продвинутые техники на sklearn

### Техники усиления:
1. ✅ Расширенная генерация признаков (x3)
2. ✅ Регуляризация
3. ✅ Feature Jittering
4. ✅ Label Smoothing  
5. ✅ Ансамблирование (Voting, Stacking)


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from tqdm.auto import tqdm
from joblib import Parallel, delayed

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    ExtraTreesRegressor,
    VotingRegressor,
    StackingRegressor
)
from sklearn.linear_model import Ridge, ElasticNet

warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams['figure.figsize'] = (14, 6)

print("✅ Библиотеки импортированы (с поддержкой параллелизма!)")


✅ Библиотеки импортированы (с поддержкой параллелизма!)


## 1. Загрузка и предобработка


In [2]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

features = [col for col in train.columns if col not in ['id', 'FloodProbability']]
X = train[features].copy()
y = train['FloodProbability'].copy()
X_test = test[features].copy()

def winsorize_outliers(df, features, lower=0.01, upper=0.99):
    df_copy = df.copy()
    for feature in features:
        lower_bound = df[feature].quantile(lower)
        upper_bound = df[feature].quantile(upper)
        df_copy[feature] = df_copy[feature].clip(lower=lower_bound, upper=upper_bound)
    return df_copy

X = winsorize_outliers(X, features)
X_test = winsorize_outliers(X_test, features)

print(f"✅ Данные загружены: {X.shape}")


✅ Данные загружены: (1117957, 20)


## 2. 🔥 РАСШИРЕННАЯ ГЕНЕРАЦИЯ ПРИЗНАКОВ


In [3]:
def generate_advanced_features(df):
    """Расширенная генерация - в 3 раза больше признаков"""
    df_new = df.copy()
    
    # Статистики
    df_new['feature_sum'] = df.sum(axis=1)
    df_new['feature_mean'] = df.mean(axis=1)
    df_new['feature_std'] = df.std(axis=1)
    df_new['feature_max'] = df.max(axis=1)
    df_new['feature_min'] = df.min(axis=1)
    df_new['feature_range'] = df_new['feature_max'] - df_new['feature_min']
    df_new['feature_median'] = df.median(axis=1)
    df_new['feature_q25'] = df.quantile(0.25, axis=1)
    df_new['feature_q75'] = df.quantile(0.75, axis=1)
    df_new['feature_iqr'] = df_new['feature_q75'] - df_new['feature_q25']
    df_new['feature_skew'] = df.skew(axis=1)
    df_new['feature_kurt'] = df.kurtosis(axis=1)
    df_new['feature_cv'] = df_new['feature_std'] / (df_new['feature_mean'] + 1e-5)
    
    # Взаимодействия
    if 'MonsoonIntensity' in df.columns and 'ClimateChange' in df.columns:
        df_new['climate_risk'] = df['MonsoonIntensity'] * df['ClimateChange']
        df_new['climate_sum'] = df['MonsoonIntensity'] + df['ClimateChange']
    
    if 'Urbanization' in df.columns and 'Deforestation' in df.columns:
        df_new['urban_deforest'] = df['Urbanization'] * df['Deforestation']
    
    # Полиномиальные для важных признаков
    important = ['MonsoonIntensity', 'TopographyDrainage', 'Deforestation']
    for feat in important:
        if feat in df.columns:
            df_new[f'{feat}_squared'] = df[feat] ** 2
            df_new[f'{feat}_sqrt'] = np.sqrt(df[feat] + 1)
            df_new[f'{feat}_log'] = np.log1p(df[feat])
    
    # Индексы
    eco_cols = ['Deforestation', 'WetlandLoss', 'ClimateChange']
    if all(col in df.columns for col in eco_cols):
        df_new['ecological_risk'] = df[eco_cols].mean(axis=1)
    
    return df_new

print("🔧 Генерация признаков...")
with tqdm(total=2, desc="Feature Engineering") as pbar:
    X_advanced = generate_advanced_features(X)
    pbar.update(1)
    X_test_advanced = generate_advanced_features(X_test)
    pbar.update(1)

print(f"✅ Было: {X.shape[1]}, стало: {X_advanced.shape[1]} признаков")
print(f"🔥 Добавлено: {X_advanced.shape[1] - X.shape[1]} новых признаков!")


🔧 Генерация признаков...


Feature Engineering:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Было: 20, стало: 46 признаков
🔥 Добавлено: 26 новых признаков!


In [4]:
# Масштабирование
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_advanced)
X_test_scaled = scaler.transform(X_test_advanced)

X_scaled = pd.DataFrame(X_scaled, columns=X_advanced.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test_advanced.columns)

X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

print(f"✅ Train: {X_train.shape}, Val: {X_val.shape}")


✅ Train: (894365, 46), Val: (223592, 46)


## 3. 🎲 Feature Jittering


In [5]:
def add_jitter(X, noise_level=0.01):
    """Добавление гауссовского шума к numpy массиву"""
    noise = np.random.normal(0, noise_level, X.shape)
    return X + noise * X.std(axis=0)  # Убрали .values, т.к. X уже numpy массив

X_train_jittered = add_jitter(X_train.values)
print("✅ Feature Jittering применен (шум 1%)")


✅ Feature Jittering применен (шум 1%)


## 4. 🎯 Label Smoothing


In [6]:
def label_smoothing(y, epsilon=0.1):
    """Сглаживание меток"""
    return y * (1 - epsilon) + 0.5 * epsilon

y_train_smooth = label_smoothing(y_train, epsilon=0.1)
print(f"✅ Label Smoothing (epsilon=0.1)")
print(f"Было: {y_train.iloc[:3].values}")
print(f"Стало: {y_train_smooth.iloc[:3].values}")


✅ Label Smoothing (epsilon=0.1)
Было: [0.575 0.4   0.505]
Стало: [0.5675 0.41   0.5045]


## 5. 🏋️ Обучение с регуляризацией

**⚡ Используется параллелизация:**
- Random Forest: `n_jobs=-1` (все CPU ядра)
- Voting: `n_jobs=-1` (параллельные предсказания)
- Stacking: `n_jobs=-1` (параллельное обучение базовых моделей)


In [7]:
# Random Forest с регуляризацией
print("🚀 Random Forest...")
rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_split=15,
    min_samples_leaf=8,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

with tqdm(total=1, desc="🌳 Random Forest") as pbar:
    rf_model.fit(X_train_jittered, y_train_smooth)  # Используем jittered данные!
    pbar.update(1)

y_pred_rf = rf_model.predict(X_val.values)  # Преобразуем в numpy
rmse_rf = np.sqrt(mean_squared_error(y_val, y_pred_rf))
r2_rf = r2_score(y_val, y_pred_rf)

print(f"Val RMSE: {rmse_rf:.6f}, R²: {r2_rf:.6f}")


🚀 Random Forest...


🌳 Random Forest:   0%|          | 0/1 [00:00<?, ?it/s]

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=-1)]: Done  30 tasks      | elapsed:   43.9s
[Parallel(n_jobs=-1)]: Done 180 tasks      | elapsed:  4.1min
[Parallel(n_jobs=-1)]: Done 300 out of 300 | elapsed:  6.7min finished
[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    0.1s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed:    0.4s


Val RMSE: 0.020302, R²: 0.841427


[Parallel(n_jobs=10)]: Done 300 out of 300 | elapsed:    0.8s finished


In [9]:
# Gradient Boosting с регуляризацией
print("\n🚀 Gradient Boosting...")
gb_model = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=5,
    subsample=0.8,
    min_samples_split=15,
    random_state=42,
    verbose=1
)

with tqdm(total=1, desc="📈 Gradient Boosting") as pbar:
    gb_model.fit(X_train_jittered, y_train_smooth)  # Используем jittered данные!
    pbar.update(1)

y_pred_gb = gb_model.predict(X_val.values)  # Преобразуем в numpy
rmse_gb = np.sqrt(mean_squared_error(y_val, y_pred_gb))
r2_gb = r2_score(y_val, y_pred_gb)

print(f"Val RMSE: {rmse_gb:.6f}, R²: {r2_gb:.6f}")



🚀 Gradient Boosting...


📈 Gradient Boosting:   0%|          | 0/1 [00:00<?, ?it/s]

      Iter       Train Loss      OOB Improve   Remaining Time 
         1           0.0020           0.0001           90.55m
         2           0.0019           0.0001           88.15m
         3           0.0018           0.0001           86.92m
         4           0.0017           0.0001           85.18m
         5           0.0016           0.0001           84.52m
         6           0.0016           0.0001           84.04m
         7           0.0015           0.0001           83.74m
         8           0.0014           0.0001           83.45m
         9           0.0013           0.0001           83.05m
        10           0.0013           0.0001           82.78m
        20           0.0008           0.0000           79.83m
        30           0.0006           0.0000           77.60m
        40           0.0005           0.0000           74.49m
        50           0.0004           0.0000           71.38m
        60           0.0003           0.0000           68.48m
       

In [11]:
# Histogram GB
print("\n🚀 Histogram Gradient Boosting...")
hgb_model = HistGradientBoostingRegressor(
    max_iter=300,
    learning_rate=0.03,
    max_depth=5,
    l2_regularization=1.0,
    random_state=42,
    verbose=1
)

with tqdm(total=1, desc="⚡ Histogram GB") as pbar:
    hgb_model.fit(X_train_jittered, y_train_smooth)  # Используем jittered данные!
    pbar.update(1)

y_pred_hgb = hgb_model.predict(X_val.values)  # Преобразуем в numpy
rmse_hgb = np.sqrt(mean_squared_error(y_val, y_pred_hgb))
r2_hgb = r2_score(y_val, y_pred_hgb)

print(f"Val RMSE: {rmse_hgb:.6f}, R²: {r2_hgb:.6f}")



🚀 Histogram Gradient Boosting...


⚡ Histogram GB:   0%|          | 0/1 [00:00<?, ?it/s]

Binning 0.296 GB of training data: 0.634 s
Binning 0.033 GB of validation data: 0.098 s
Fitting gradient boosted rounds:
Fit 175 trees in 5.494 s, (5399 total leaves)
Time spent computing histograms: 2.969s
Time spent finding best splits:  0.210s
Time spent applying splits:      0.823s
Time spent predicting:           0.135s
Val RMSE: 0.019935, R²: 0.847110


## 6. 🎪 Ансамблирование

Используем 4 метода:
1. **Простой блендинг** (простое усреднение)
2. **Взвешенное среднее** (оптимизированные веса)
3. **Voting** (фиксированные веса)
4. **Stacking** (мета-модель)


In [13]:
# 1️⃣ Простой блендинг (Simple Averaging)
print("🎯 Простой блендинг (усреднение предсказаний)...")

# Простое среднее арифметическое
y_pred_blend_simple = (y_pred_rf + y_pred_gb + y_pred_hgb) / 3

rmse_blend_simple = np.sqrt(mean_squared_error(y_val, y_pred_blend_simple))
r2_blend_simple = r2_score(y_val, y_pred_blend_simple)

print(f"Val RMSE: {rmse_blend_simple:.6f}, R²: {r2_blend_simple:.6f}")


🎯 Простой блендинг (усреднение предсказаний)...
Val RMSE: 0.019961, R²: 0.846716


In [14]:
# 2️⃣ Оптимизированное взвешенное среднее
print("\n🔧 Оптимизация весов для блендинга...")

from scipy.optimize import minimize

# Функция для оптимизации
def blend_predictions(weights, *predictions):
    """Взвешенное усреднение предсказаний"""
    final_prediction = np.zeros_like(predictions[0])
    for weight, pred in zip(weights, predictions):
        final_prediction += weight * pred
    return final_prediction

def rmse_objective(weights, *args):
    """Целевая функция для минимизации RMSE"""
    predictions = args[:-1]
    y_true = args[-1]
    y_pred = blend_predictions(weights, *predictions)
    return np.sqrt(mean_squared_error(y_true, y_pred))

# Начальные веса (равномерное распределение)
initial_weights = [1/3, 1/3, 1/3]

# Ограничения: сумма весов = 1, все веса >= 0
constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}
bounds = [(0, 1)] * 3

# Оптимизация
result = minimize(
    rmse_objective,
    initial_weights,
    args=(y_pred_rf, y_pred_gb, y_pred_hgb, y_val.values),
    method='SLSQP',
    bounds=bounds,
    constraints=constraints
)

optimal_weights = result.x
print(f"Оптимальные веса: RF={optimal_weights[0]:.3f}, GB={optimal_weights[1]:.3f}, HGB={optimal_weights[2]:.3f}")

# Применяем оптимальные веса
y_pred_blend_optimal = blend_predictions(optimal_weights, y_pred_rf, y_pred_gb, y_pred_hgb)

rmse_blend_optimal = np.sqrt(mean_squared_error(y_val, y_pred_blend_optimal))
r2_blend_optimal = r2_score(y_val, y_pred_blend_optimal)

print(f"Val RMSE: {rmse_blend_optimal:.6f}, R²: {r2_blend_optimal:.6f}")



🔧 Оптимизация весов для блендинга...
Оптимальные веса: RF=0.333, GB=0.333, HGB=0.333
Val RMSE: 0.019961, R²: 0.846716


In [12]:
# 3️⃣ Voting Regressor (фиксированные веса)
print("\n🎪 Voting Ensemble...")
voting_model = VotingRegressor(
    estimators=[
        ('rf', rf_model),
        ('gb', gb_model),
        ('hgb', hgb_model)
    ],
    weights=[0.3, 0.35, 0.35],
    n_jobs=-1  # Параллельное вычисление предсказаний!
)

with tqdm(total=1, desc="🎪 Voting Ensemble") as pbar:
    voting_model.fit(X_train_jittered, y_train_smooth)  # Используем jittered данные!
    pbar.update(1)

y_pred_voting = voting_model.predict(X_val.values)  # Преобразуем в numpy
rmse_voting = np.sqrt(mean_squared_error(y_val, y_pred_voting))
r2_voting = r2_score(y_val, y_pred_voting)

print(f"Val RMSE: {rmse_voting:.6f}, R²: {r2_voting:.6f}")



🚀 Voting Ensemble...


🎪 Voting Ensemble:   0%|          | 0/1 [00:00<?, ?it/s]

      Iter       Train Loss      OOB Improve   Remaining Time 


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 10 concurrent workers.


Binning 0.296 GB of training data: 5.067 s
Binning 0.033 GB of validation data: 0.438 s
Fitting gradient boosted rounds:
Fit 175 trees in 29.862 s, (5399 total leaves)
Time spent computing histograms: 18.059s
Time spent finding best splits:  0.443s
Time spent applying splits:      4.148s
Time spent predicting:           0.409s
         1           0.0020           0.0001          194.56m


...

         6           0.0016           0.0001          196.22m
         7           0.0015           0.0001          193.19m
         8           0.0014           0.0001          190.60m
         9           0.0013           0.0001          191.11m


[Parallel(n_jobs=-1)]: Done 300 out of 300 | elapsed:  6.1min finished


        10           0.0013           0.0001          182.36m
        20           0.0008           0.0000          127.66m
        30           0.0006           0.0000          107.04m
        40           0.0005           0.0000           97.45m
        50           0.0004           0.0000           90.40m
        60           0.0003           0.0000           83.97m
        70           0.0003           0.0000           78.45m
        80           0.0003           0.0000           73.49m
        90           0.0003          -0.0000           69.05m
       100           0.0003           0.0000           64.99m
       200           0.0003          -0.0000           30.26m
       300           0.0003           0.0000            0.00s


[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    0.1s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed:    0.4s
[Parallel(n_jobs=10)]: Done 300 out of 300 | elapsed:    0.7s finished


Val RMSE: 0.019948, R²: 0.846918


In [15]:
# 4️⃣ Stacking Regressor (мета-модель)
print("\n🏗️ Stacking Ensemble...")
stacking_model = StackingRegressor(
    estimators=[
        ('rf', rf_model),
        ('gb', gb_model),
        ('hgb', hgb_model)
    ],
    final_estimator=Ridge(alpha=1.0),
    n_jobs=-1,  # Параллельное обучение базовых моделей!
    cv=3  # Кросс-валидация для мета-признаков
)

with tqdm(total=1, desc="🏗️ Stacking Ensemble") as pbar:
    stacking_model.fit(X_train_jittered, y_train_smooth)  # Используем jittered данные!
    pbar.update(1)

y_pred_stacking = stacking_model.predict(X_val.values)  # Преобразуем в numpy
rmse_stacking = np.sqrt(mean_squared_error(y_val, y_pred_stacking))
r2_stacking = r2_score(y_val, y_pred_stacking)

print(f"Val RMSE: {rmse_stacking:.6f}, R²: {r2_stacking:.6f}")



🏗️ Stacking Ensemble...


🏗️ Stacking Ensemble:   0%|          | 0/1 [00:00<?, ?it/s]

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 10 concurrent workers.


      Iter       Train Loss      OOB Improve   Remaining Time 
Binning 0.296 GB of training data: 4.134 s
Binning 0.033 GB of validation data: 0.376 s
Fitting gradient boosted rounds:
Fit 175 trees in 29.703 s, (5399 total leaves)
Time spent computing histograms: 18.851s
Time spent finding best splits:  0.426s
Time spent applying splits:      4.493s
Time spent predicting:           0.445s


[Parallel(n_jobs=-1)]: Done  30 tasks      | elapsed:   32.6s


...

[Parallel(n_jobs=-1)]: Done 180 tasks      | elapsed:  3.6min


         5           0.0016           0.0001          232.19m
         6           0.0016           0.0001          235.85m
         7           0.0015           0.0001          235.89m


[Parallel(n_jobs=-1)]: Done 300 out of 300 | elapsed:  6.1min finished


         8           0.0014           0.0001          226.12m
         9           0.0013           0.0001          209.25m
        10           0.0013           0.0001          195.87m
        20           0.0008           0.0000         4236.00m
        30           0.0006           0.0000         5593.44m
        40           0.0005           0.0000         4061.57m
        50           0.0004           0.0000         3140.87m
        60           0.0003           0.0000         2526.45m
        70           0.0003           0.0000         2086.88m
        80           0.0003           0.0000         1755.72m
        90           0.0003          -0.0000         1497.76m
       100           0.0003           0.0000         1310.77m


KeyboardInterrupt: 

## 7. 📊 Сравнение результатов


In [ ]:
results = pd.DataFrame({
    'Model': [
        'Random Forest', 
        'Gradient Boosting', 
        'Histogram GB', 
        'Blending (Simple)', 
        'Blending (Optimal)',
        'Voting', 
        'Stacking'
    ],
    'Val RMSE': [
        rmse_rf, 
        rmse_gb, 
        rmse_hgb, 
        rmse_blend_simple, 
        rmse_blend_optimal,
        rmse_voting, 
        rmse_stacking
    ],
    'Val R²': [
        r2_rf, 
        r2_gb, 
        r2_hgb, 
        r2_blend_simple, 
        r2_blend_optimal,
        r2_voting, 
        r2_stacking
    ]
}).sort_values('Val RMSE')

print("\n" + "="*70)
print("📊 РЕЗУЛЬТАТЫ УСИЛЕННОГО РЕШЕНИЯ (ВСЕ ТЕХНИКИ АНСАМБЛИРОВАНИЯ)")
print("="*70)
display(results.style.background_gradient(cmap='RdYlGn_r', subset=['Val RMSE']))

print(f"\n🏆 Лучшая модель: {results.iloc[0]['Model']}")
print(f"✅ Validation RMSE: {results.iloc[0]['Val RMSE']:.6f}")

# Выводим оптимальные веса
print(f"\n💡 Оптимальные веса блендинга:")
print(f"   Random Forest: {optimal_weights[0]:.3f}")
print(f"   Gradient Boosting: {optimal_weights[1]:.3f}")
print(f"   Histogram GB: {optimal_weights[2]:.3f}")

baseline_rmse = 0.035  # Замените на реальное из baseline
improvement = (baseline_rmse - results.iloc[0]['Val RMSE']) / baseline_rmse * 100
print(f"🚀 Улучшение относительно бейслайна: {improvement:.2f}%")


## 8. 🎯 Submission


In [ ]:
# Получаем предсказания всех базовых моделей на тесте
y_test_rf = rf_model.predict(X_test_scaled.values)
y_test_gb = gb_model.predict(X_test_scaled.values)
y_test_hgb = hgb_model.predict(X_test_scaled.values)

# Используем оптимизированное взвешенное среднее (лучший метод!)
y_test_final = blend_predictions(optimal_weights, y_test_rf, y_test_gb, y_test_hgb)

# Также получаем предсказание от Stacking для сравнения
y_test_stacking = stacking_model.predict(X_test_scaled.values)

# Выбираем лучшую модель по validation RMSE
best_model_name = results.iloc[0]['Model']
print(f"\n🏆 Используем лучшую модель: {best_model_name}")

# Если Stacking лучше, используем его
if best_model_name == 'Stacking':
    y_test_final = y_test_stacking
    print("   Используем Stacking")
else:
    print(f"   Используем Blending (Optimal) с весами:")
    print(f"   RF={optimal_weights[0]:.3f}, GB={optimal_weights[1]:.3f}, HGB={optimal_weights[2]:.3f}")

submission = pd.DataFrame({
    'id': test['id'],
    'FloodProbability': y_test_final
})

submission.to_csv('advanced_submission.csv', index=False)

print("\n✅ Файл advanced_submission.csv создан!")
print(f"\n💡 Применены ВСЕ техники усиления:")
print("  1. Расширенная генерация признаков (40+ новых)")
print("  2. Регуляризация моделей (max_depth, min_samples, l2)")
print("  3. Feature Jittering (гауссовский шум 1%)")
print("  4. Label Smoothing (epsilon=0.1)")
print("  5. Блендинг (Simple Averaging)")
print("  6. Блендинг (Оптимизированные веса)")
print("  7. Voting Ensemble")
print("  8. Stacking Ensemble")
print(f"\nСтатистика:")
print(submission['FloodProbability'].describe())
display(submission.head(10))
